# 01 — Score NSD Images

This notebook computes two scores for each NSD image:
1. **Visual Stimulation Load** — Shannon entropy of the luminance histogram
2. **Conceptual Abstractness** — inverse of mean concreteness (Brysbaert norms) applied to COCO captions

Both scores are z-scored across the full corpus before downstream use.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Ensure scripts are importable
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

sns.set_style('whitegrid')
sns.set_context('notebook')
%matplotlib inline

## Configuration

Set your local paths here.

In [ ]:
# ======================== EDIT THESE PATHS ========================
NSD_IMAGE_DIR = '/path/to/nsd/stimuli/nsd'            # Directory of NSD stimulus images
COCO_CAPTIONS = '/path/to/coco/captions_train2017.json' # COCO captions JSON
BRYSBAERT_FILE = 'data/brysbaert_2014_concreteness.xlsx' # Brysbaert norms
NSD_STIM_INFO = '/path/to/nsd_stim_info_merged.csv'    # NSD stim info (nsd_id -> coco_id)
# =================================================================

OUTPUT_SCORES = 'data/nsd_image_scores.csv'

## Step 1: Score Visual Load (Entropy)

In [ ]:
from score_load import image_entropy, mean_luminance, rms_contrast, edge_density
from PIL import Image

# Discover images
image_dir = Path(NSD_IMAGE_DIR)
image_paths = sorted(
    [p for ext in ['*.png', '*.jpg', '*.jpeg'] for p in image_dir.glob(ext)]
)
print(f'Found {len(image_paths)} images')

In [ ]:
# Compute load metrics for all images
from concurrent.futures import ProcessPoolExecutor, as_completed
from score_load import compute_all_metrics

records = []
with ProcessPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(compute_all_metrics, str(p)): p for p in image_paths}
    for i, future in enumerate(as_completed(futures), 1):
        try:
            records.append(future.result())
        except Exception as e:
            print(f'Failed: {futures[future]}: {e}')
        if i % 10000 == 0:
            print(f'  {i}/{len(image_paths)}')

df = pd.DataFrame(records).sort_values('nsd_id').reset_index(drop=True)
print(f'Scored {len(df)} images')
df.head()

In [ ]:
# Z-score entropy
df['load_z'] = (df['entropy'] - df['entropy'].mean()) / df['entropy'].std()

# Z-score nuisance metrics
for col in ['mean_luminance', 'rms_contrast', 'edge_density']:
    if col in df.columns and df[col].notna().sum() > 0:
        m, s = df[col].mean(), df[col].std()
        if s > 0:
            df[f'{col}_z'] = (df[col] - m) / s

print('Load score distribution:')
df[['entropy', 'load_z']].describe()

### Visualize load score distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['entropy'].dropna(), bins=80, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Shannon Entropy')
axes[0].set_ylabel('Count')
axes[0].set_title('Visual Load (Entropy)')

axes[1].hist(df['mean_luminance'].dropna(), bins=80, color='gray', edgecolor='white')
axes[1].set_xlabel('Mean Luminance')
axes[1].set_title('Mean Luminance')

axes[2].hist(df['rms_contrast'].dropna(), bins=80, color='coral', edgecolor='white')
axes[2].set_xlabel('RMS Contrast')
axes[2].set_title('RMS Contrast')

plt.tight_layout()
plt.show()

## Step 2: Score Abstractness (Brysbaert Norms × COCO Captions)

In [ ]:
from score_abstractness import load_brysbaert, load_coco_captions, score_image_abstractness
from nltk.corpus import stopwords
import nltk

# Download NLTK data if needed
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load resources
norms = load_brysbaert(BRYSBAERT_FILE)
coco_caps = load_coco_captions(COCO_CAPTIONS)
stops = set(stopwords.words('english'))

print(f'Brysbaert norms: {len(norms)} words')
print(f'COCO captions: {len(coco_caps)} images')

In [ ]:
# Map nsd_id -> coco_id if needed
if 'coco_id' not in df.columns:
    from score_abstractness import load_nsd_to_coco
    nsd2coco = load_nsd_to_coco(NSD_STIM_INFO)
    df['coco_id'] = df['nsd_id'].map(nsd2coco)

# Score abstractness
abstractness = []
for _, row in df.iterrows():
    coco_id = row.get('coco_id')
    if pd.isna(coco_id):
        abstractness.append(np.nan)
        continue
    captions = coco_caps.get(int(coco_id), [])
    score = score_image_abstractness(captions, norms, stops, min_words=3)
    abstractness.append(score if score is not None else np.nan)

df['abstractness'] = abstractness

# Z-score
valid = df['abstractness'].notna()
df['abstractness_z'] = np.nan
df.loc[valid, 'abstractness_z'] = (
    (df.loc[valid, 'abstractness'] - df.loc[valid, 'abstractness'].mean())
    / df.loc[valid, 'abstractness'].std()
)

n_valid = valid.sum()
print(f'Scored {n_valid}/{len(df)} images on abstractness')
df[['abstractness', 'abstractness_z']].describe()

### Visualize abstractness distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['abstractness'].dropna(), bins=80, color='mediumpurple', edgecolor='white')
axes[0].set_xlabel('Abstractness (5 - mean concreteness)')
axes[0].set_ylabel('Count')
axes[0].set_title('Abstractness Distribution')

# Scatter: load vs abstractness
both_valid = df['load_z'].notna() & df['abstractness_z'].notna()
axes[1].scatter(
    df.loc[both_valid, 'load_z'],
    df.loc[both_valid, 'abstractness_z'],
    alpha=0.05, s=3, color='teal'
)
axes[1].set_xlabel('Load (z)')
axes[1].set_ylabel('Abstractness (z)')
axes[1].set_title('Load vs Abstractness')

from scipy.stats import pearsonr
r, p = pearsonr(
    df.loc[both_valid, 'load_z'],
    df.loc[both_valid, 'abstractness_z']
)
axes[1].text(0.05, 0.95, f'r = {r:.3f}\np = {p:.2e}',
             transform=axes[1].transAxes, va='top',
             fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Save Scores

In [ ]:
os.makedirs('data', exist_ok=True)
df.to_csv(OUTPUT_SCORES, index=False)
print(f'Saved {len(df)} image scores to {OUTPUT_SCORES}')
print(f'Columns: {list(df.columns)}')

### Example images at the extremes

In [ ]:
# Show example images at corners of the load-abstractness space
both_valid = df['load_z'].notna() & df['abstractness_z'].notna()
df_valid = df[both_valid].copy()

corners = {
    'Low Load, Concrete': df_valid.nsmallest(5, 'load_z').nsmallest(1, 'abstractness_z'),
    'Low Load, Abstract': df_valid.nsmallest(5, 'load_z').nlargest(1, 'abstractness_z'),
    'High Load, Concrete': df_valid.nlargest(5, 'load_z').nsmallest(1, 'abstractness_z'),
    'High Load, Abstract': df_valid.nlargest(5, 'load_z').nlargest(1, 'abstractness_z'),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (label, sub_df) in zip(axes, corners.items()):
    row = sub_df.iloc[0]
    img_path = row.get('img_path', '')
    if os.path.exists(str(img_path)):
        img = Image.open(img_path)
        ax.imshow(img)
    ax.set_title(f"{label}\nload_z={row['load_z']:.2f}\nabst_z={row['abstractness_z']:.2f}",
                 fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()